In [7]:
# BASE모델 , 파인튜닝 Instruct 성능 비교
# Qween2.5  Qween2.5_instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = '프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘'
base_model_id = 'Qwen/Qwen2.5-0.5B'
instruct_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

def generate(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.1,
        do_sample=True,
        max_new_tokens=40
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [9]:
print(f'질문 : {prompt}')
base_output = generate(base_model_id)
print(f'[Base 모델 출력 : {base_output}]')
instruct_output = generate(instruct_model_id)
print(f'[Instruct 모델 출력 : {instruct_output}]')

질문 : 프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Base 모델 출력 : 프랑스 파리의 명소를 추천해드리겠습니다. 1. 레이크 파리 (Rêche Paris) - 레이크 파리는]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Instruct 모델 출력 : 프랑스의 파리는 많은 관광지가 있습니다. 아래에 두 가지 명소를 추천드립니다:

1. 루이스 아트로 (Rue de Riv]


In [ ]:
# 정량적 평가 BLUE, ROUGE
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    blue_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    rouge_f1 = scores['rouge-1']['f']
    return scores,rouge_f1

ref = '파리의 명소로는 에펠탑과 루브르 박물관을 추천합니다.'
b_blue, b_rouge = evaluate_ngram(ref,base_output)
i_blue, i_rouge = evaluate_ngram(ref,instruct_output)

print(f'Base모델 BLUE : {b_blue} ROUGE-1 : {b_rouge}')
print(f'Instruct모델 BLUE : {i_blue} ROUGE-1 : {i_rouge}')